<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/DayChallenge_week6_day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Étape 0 : Préparation de l'environnement et exploration des données

Avant de commencer les tâches, nous devons décompresser les fichiers et charger les données pour comprendre leur structure.

In [6]:
import zipfile
import os
import pandas as pd

# Recherche automatique d'un fichier ZIP si le chemin exact échoue
def find_and_extract(extract_to='/content/dataset_nlp'):
    content_files = os.listdir('/content')
    zips = [f for f in content_files if f.endswith('.zip')]
    if zips:
        target_zip = os.path.join('/content', zips[0])
        print(f"Fichier trouvé : {target_zip}")
        with zipfile.ZipFile(target_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        return True
    return False

extract_path = '/content/dataset_nlp'
if find_and_extract(extract_path):
    files = []
    for root, dirs, filenames in os.walk(extract_path):
        for f in filenames: files.append(os.path.join(root, f))
    csv_files = [f for f in files if f.endswith('.csv')]
    if csv_files:
        df = pd.read_csv(csv_files[0])
        print("Données chargées.")
        display(df.head())
    else:
        df = pd.DataFrame({'text': ['Exemple 1', 'Exemple 2'], 'label': [0, 1]})
else:
    print("Aucun ZIP trouvé. Utilisation de données fictives.")
    df = pd.DataFrame({'text': ['Exemple 1', 'Exemple 2'], 'label': [0, 1]})

Aucun ZIP trouvé. Utilisation de données fictives.


### Question 1 : Comprendre BERT et XLM-RoBERTa

*   **BERT (Bidirectional Encoder Representations from Transformers)** : Pré-entraîné sur du texte non étiqueté en utilisant le masquage de mots. Il est bidirectionnel, ce qui signifie qu'il regarde le contexte à gauche et à droite d'un mot.
*   **XLM-RoBERTa** : Une variante multilingue de RoBERTa (une version optimisée de BERT) entraînée sur un volume massif de données multilingues (CommonCrawl). Elle est particulièrement efficace pour les tâches impliquant plusieurs langues.

In [2]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Initialisation des tokeniseurs
try:
    bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
    print("Tokeniseurs chargés avec succès.")
except Exception as e:
    print(f"Erreur lors du chargement des tokeniseurs : {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokeniseurs chargés avec succès.


### Question 2 : Tokenisation du texte

*   Utilisation de `encode_plus` pour obtenir les `input_ids` et `attention_mask`.
*   Décodage des tokens pour vérifier la reconstruction du texte.
*   Exemple de tokenisation simple et par paire.

In [7]:
sentence1 = "Ceci est un défi quotidien."
sentence2 = "Les transformateurs sont puissants."

# Utilisation de l'appel direct au lieu de encode_plus
bert_output = bert_tokenizer(
    sentence1,
    sentence2,
    add_special_tokens=True,
    return_attention_mask=True,
    return_tensors='pt'
)

xlm_output = xlm_tokenizer(
    sentence1,
    add_special_tokens=True,
    return_attention_mask=True,
    return_tensors='pt'
)

print("BERT Tokens IDs:", bert_output['input_ids'])
print("BERT Décodage:", bert_tokenizer.decode(bert_output['input_ids'][0]))
print("\nXLM-RoBERTa Tokens IDs:", xlm_output['input_ids'])
print("XLM-RoBERTa Décodage:", xlm_tokenizer.decode(xlm_output['input_ids'][0]))

BERT Tokens IDs: tensor([[  101,  8292,  6895,  9765,  4895, 13366,  2072, 22035,  3775, 10265,
          2078,  1012,   102,  4649, 10938,  3686,  9236,  2365,  2102, 16405,
         21205,  7666,  1012,   102]])
BERT Décodage: [CLS] ceci est un defi quotidien. [SEP] les transformateurs sont puissants. [SEP]

XLM-RoBERTa Tokens IDs: tensor([[     0, 124532,    437,     51, 100896, 105036,      5,      2]])
XLM-RoBERTa Décodage: <s> Ceci est un défi quotidien.</s>


### Question 3 : Préparation des données d'entrée pour le modèle

*   Configuration de la longueur maximale (`max_length`).
*   Ajout de remplissage (`padding`) et troncature (`truncation`).
*   Exploration des jetons spéciaux et de la taille du vocabulaire.

In [8]:
max_len = 32

# Appel direct pour la préparation complète
encoded_inputs = xlm_tokenizer(
    sentence1,
    max_length=max_len,
    padding='max_length',
    truncation=True,
    add_special_tokens=True,
    return_attention_mask=True
)

print(f"Taille du vocabulaire XLM-R: {xlm_tokenizer.vocab_size}")
print(f"Jetons spéciaux XLM-R: {xlm_tokenizer.special_tokens_map}")
print(f"Attention Mask (longueur {max_len}):", encoded_inputs['attention_mask'])

Taille du vocabulaire XLM-R: 250002
Jetons spéciaux XLM-R: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
Attention Mask (longueur 32): [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### Question 4 : Chargement et exploration de l'ensemble de données

*   Chargement du DataFrame.
*   Inspection de la structure (shape, colonnes, types).

In [9]:
print(f"Dimensions de l'ensemble de données : {df.shape}")
print("\nColonnes disponibles :", df.columns.tolist())
print("\nAperçu des données :")
display(df.head())

# Vérification des valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())

Dimensions de l'ensemble de données : (2, 2)

Colonnes disponibles : ['text', 'label']

Aperçu des données :


,text,label
0,Exemple 1,0
1,Exemple 2,1



Valeurs manquantes par colonne :
text     0
label    0
dtype: int64


### Question 5 : Création de plis de validation croisée

*   Utilisation de `StratifiedKFold` pour diviser les données en 5 plis.
*   Maintien de la distribution des étiquettes dans chaque pli.

In [10]:
from sklearn.model_selection import StratifiedKFold

# Initialisation du K-Fold stratifié (5 plis)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_folds = []
val_folds = []

# Simulation d'indices (si df est petit, ajuster n_splits ou utiliser plus de données)
try:
    for fold_id, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
        train_folds.append(df.iloc[train_idx])
        val_folds.append(df.iloc[val_idx])
        print(f"Pli {fold_id + 1} créé : Train={len(train_idx)}, Val={len(val_idx)}")
except ValueError as e:
    print(f"Erreur lors du split (possiblement trop peu de données) : {e}")

print(f"\nNombre total de plis stockés : {len(train_folds)}")

Erreur lors du split (possiblement trop peu de données) : Cannot have number of splits n_splits=5 greater than the number of samples: n_samples=2.

Nombre total de plis stockés : 0


### Script Final Consolidé

Ce script regroupe toutes les étapes du défi : exploration, tokenisation BERT/XLM-R, préparation des données et validation croisée.

In [11]:
import os
import zipfile
import pandas as pd
import torch
from transformers import BertTokenizer, XLMRobertaTokenizer
from sklearn.model_selection import StratifiedKFold

# 1. Chargement et Exploration
def prepare_data():
    extract_path = '/content/dataset_nlp'
    content_files = os.listdir('/content')
    zips = [f for f in content_files if f.endswith('.zip')]

    if zips:
        with zipfile.ZipFile(os.path.join('/content', zips[0]), 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        for root, _, filenames in os.walk(extract_path):
            for f in filenames:
                if f.endswith('.csv'):
                    return pd.read_csv(os.path.join(root, f))

    print("Utilisation de données fictives car aucun CSV n'a été trouvé.")
    return pd.DataFrame({
        'text': [f'Texte exemple {i}' for i in range(20)],
        'label': [0, 1] * 10
    })

df_final = prepare_data()
print(f"Données chargées : {df_final.shape}")

# 2. Tokenisation
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

text_sample = "Apprendre les Transformers est passionnant."

# BERT
bert_input = bert_tokenizer(text_sample, padding='max_length', max_length=16, truncation=True, return_tensors='pt')
# XLM-R
xlm_input = xlm_tokenizer(text_sample, padding='max_length', max_length=16, truncation=True, return_tensors='pt')

print("BERT Token IDs:", bert_input['input_ids'])
print("XLM-R Token IDs:", xlm_input['input_ids'])

# 3. Validation Croisée (K-Fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (t_idx, v_idx) in enumerate(skf.split(df_final, df_final['label'])):
    print(f"Fold {fold+1} | Train size: {len(t_idx)} | Val size: {len(v_idx)}")

Utilisation de données fictives car aucun CSV n'a été trouvé.
Données chargées : (20, 2)
BERT Token IDs: tensor([[  101, 10439,  7389, 16200,  4649, 19081,  9765,  6896, 16885,  1012,
           102,     0,     0,     0,     0,     0]])
XLM-R Token IDs: tensor([[    0,  5787, 81282,   199, 11062, 82772,     7,   437, 36096, 30125,
             5,     2,     1,     1,     1,     1]])
Fold 1 | Train size: 16 | Val size: 4
Fold 2 | Train size: 16 | Val size: 4
Fold 3 | Train size: 16 | Val size: 4
Fold 4 | Train size: 16 | Val size: 4
Fold 5 | Train size: 16 | Val size: 4
